# Demo de Inferență NUC-Net & Alpine Pentru Runtime Colab
Acest notebook demonstrează cum se încărcă modelul NUC-Net (Segmentare Semantică) distilat, se încărcă ponderile pre-antrenate, se efectuează o inferență pe o scenă reală de ansamblu de puncte LiDAR și se rulează capl de clusterizare de instanțe Alpine.

In [1]:
# Aceasta seciune este destinata utilizarii in interiorul browser-ului, cu mediul/runtime colab
# si necesita un cont google valid. Pentru a folosi cu runtime local, trebuie folosit scriptul alternativ
# `local_inference_demo.ipynb` pentru care trebuie realizate setarile si configuratiile necesare descrise
# in README.md
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import os
import sys
import zipfile

# Authentificarea utilizatorului (prin Google login pop-up)
print("Realizati autentificarea pentru a descarca securizat mediul: ")
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Definirea dictionarului cu fisierele ce necesita descarcate
dirs = [
    {
        'name' : 'libs',
        'id': '1WLchsJ_oh0rF-42ikv6PPiJksa4PItik',
        'path': '/content/libs.zip'
    },
    {
        'name' : 'data',
        'id': '1cwUzp-Km6zz2bE5mp8XZ560PCmxxCfGG',
        'path': '/content/data.zip'
    }
]

# Descarcarea prin Google Drive API oficial
for dir in dirs:
  print(f"Descarcarea {dir['name']}...")
  request = drive_service.files().get_media(fileId=dir['id'])
  downloaded = io.BytesIO()
  downloader = MediaIoBaseDownload(downloaded, request)
  done = False
  while done is False:
      status, done = downloader.next_chunk()
      print(f"Descarcat {int(status.progress() * 100)}%.")

  with open(dir['path'], 'wb') as f:
      f.write(downloaded.getvalue())

  # Extract and load the modules
  print(f"Extragerea fisierelor pentru {dir['name']}...")
  with zipfile.ZipFile(dir['path'], 'r') as zip_ref:
      zip_ref.extractall('/content/')

sys.path.insert(0, os.path.abspath('/content/libs/nucnet'))
sys.path.insert(0, os.path.abspath('/content/libs/alpine'))
print("Modulele s-au incarcat cu succes!")

Realizati autentificarea pentru a descarca securizat mediul: 


Descarcarea libs...
Descarcat 100%.
Extragerea fisierelor pentru libs...
Descarcarea data...
Descarcat 100%.
Extragerea fisierelor pentru data...
Modulele s-au incarcat cu succes!


In [2]:
#@title Descărcarea dependințelor

# Din cauza la runtime-ul colab care a fost actualizat
# Lipsesc anumite dependinte de care avem nevoie
import torch
import os

# Se va parsa versiunile exacte Colab pentru PyTorch si CUDA
def format_pytorch_version(version):
  return version.split('+')[0]

def format_cuda_version(version):
  return 'cu' + version.replace('.', '')

TORCH = format_pytorch_version(torch.__version__)
CUDA = format_cuda_version(torch.version.cuda)

print(f"Instalarea ['torch-scatter', 'spconv'] pentru PyTorch {TORCH} si CUDA {CUDA}...")

# Instalam binarele pre-compilate direct de la PyG (PyTorch Geometric) pentru a micsora timpul de asteptare
# la compilarea lor JIT din cauza runtime-ului nou al colab-ului
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

# IMPORTANT: spconv-cu126 nu are binare precompilate pentru CUDA 12.8 (versiunea
# curenta de Colab), asa ca cumm va face fallback la o compilare JIT completa
# (acesta este motivul celor 3-4 ore). Ii restrangem tinta doar la arhitectura
# GPU-ului alocat acum, in loc sa compileze pentru o gama larga de arhitecturi.
cc_major, cc_minor = torch.cuda.get_device_capability(0)
os.environ["CUMM_CUDA_ARCH_LIST"] = f"{cc_major}.{cc_minor}"
print(f"GPU detectat: capability {cc_major}.{cc_minor} -> CUMM_CUDA_ARCH_LIST={os.environ['CUMM_CUDA_ARCH_LIST']}")

# Instalam spconv-cu126 care este complet compatibil cu versiunea CUDA 12.8
!pip install spconv-cu126

# Clonam repositoriul oficial cumm in folder temporar pentru a prelua header-ele lipsa
!git clone https://github.com/FindDefinition/cumm.git /tmp/cumm

# Copiem header-ele 'tensorview' lipsa in directorul standard include al Colab-ului
# Aceasta asigura ca compilatorul gcc automat le va gasi in timpul compilarii JIT
!cp -r /tmp/cumm/include/tensorview /usr/local/include/

# La fel le copiem in directorul pachetului cumm pentru a ne asigura
# (folderul 'build/core_cc/include' nu exista inca la acest punct - este creat
# abia cand JIT-ul porneste efectiv compilarea la primul import de mai jos -
# asa ca il cream noi dinainte, ca sa nu esueze silentios copierea)
!mkdir -p /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/
!cp -r /tmp/cumm/include/tensorview /usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/

# --- Cache JIT pe Google Drive, ca sa nu recompilam de la zero la fiecare sesiune ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CACHE_DIR = "/content/drive/MyDrive/nucnet_alpine_jit_cache"
DIST_PACKAGES = "/usr/local/lib/python3.12/dist-packages"
CUMM_PKG = f"{DIST_PACKAGES}/cumm"
SPCONV_PKG = f"{DIST_PACKAGES}/spconv"
os.makedirs(CACHE_DIR, exist_ok=True)

cache_tag = f"{CACHE_DIR}/{CUDA}_{cc_major}{cc_minor}"
# NOTA: cumm/spconv folosesc pccm+ccimport pentru compilarea JIT. Acestea compileaza
# in build/<modul>/, dar apoi COPIAZA binarul .so final direct in radacina pachetului
# (ex: cumm/core_cc....so), fara a lasa o copie utilizabila in build/. De aceea cache-uim
# intregul director al pachetului, nu doar subdirectorul 'build/'.
cumm_cache = f"{cache_tag}_cumm_pkg.tar"
spconv_cache = f"{cache_tag}_spconv_pkg.tar"

JIT_CACHE_RESTORED = os.path.exists(cumm_cache) and os.path.exists(spconv_cache)

if JIT_CACHE_RESTORED:
    print("Cache JIT gasit pe Drive pentru aceasta combinatie CUDA/GPU -> se restaureaza binarele compilate...")
    !tar -xf {cumm_cache} -C {DIST_PACKAGES}
    !tar -xf {spconv_cache} -C {DIST_PACKAGES}
    print("Restaurare completa. Compilarea JIT ar trebui sa fie sarita la importul de mai jos.")
else:
    print("Niciun cache JIT potrivit gasit pe Drive.")
    print("La importul din celula urmatoare va rula compilarea completa (mai rapida acum, datorita CUMM_CUDA_ARCH_LIST).")
    print("Rezultatul va fi salvat automat pe Drive pentru sesiunile viitoare.")

print("Instalarea completa.")


Instalarea ['torch-scatter', 'spconv'] pentru PyTorch 2.11.0 si CUDA cu128...
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 109.4 MB/s eta 0:00:00
GPU detectat: capability 7.5 -> CUMM_CUDA_ARCH_LIST=7.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.3 MB/s eta 0:00:00
Cloning into '/tmp/cumm'...
remote: Enumerating objects: 3974, done.
remote: Counting objects: 100% (1292/1292), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 3974 (delta 1206), reused 1125 (delta 1125), pack-reuse

In [3]:
# Setarea mediului

# Din cauza faptului ca toate binarile si dependintele pentru colab
# se aloca pe o masina virtuala temporara, la fiecare prima rulare
# se va astepta un timp considerabil (3-4 ore) pentru compilarea
# binarelor de care se folosesc alpine si nucnet, aceasta celula
# va dura extrem de mult la prima rulare
import os
import sys
import yaml
import time
import torch
import numpy as np
import plotly.graph_objects as go

from models.student import build_student
from dataloader.dataset_semantickitti import cylinder_dataset, collate_fn_BEV
from alpine import Alpine
from alpine_semantickitti import THING_CLASSES, BBOX_WEB, CLASSES

[1/41] [GCC][c++]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/src/tensorview_bind/PyBindAppleMetalImpl/PyBindAppleMetalImpl_bind_AppleMetalImpl.cc.o
[2/41] [GCC][c++/pch]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/csrc/arrayref/ArrayPtr.h.gch
/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/csrc/arrayref/ArrayPtr.h:1:9: warning: #pragma once in main file
    1 | #pragma once
      |         ^~~~
[3/41] [GCC][c++/pch]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h.gch
/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/include/tensorview_bind/TensorViewBind.h:1:9: warning: #pragma once in main file
    1 | #pragma once
      |         ^~~~
[4/41] [GCC][c++]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/src/tensorview_bind/TensorViewBind/TensorViewBind_hello.cc.o
[5/41] [GCC][c++]/usr/local/lib/python3.12/dist-packages/cumm/build/core_cc/src/csrc/arrayref/ArrayPtr

In [4]:
#@title Salvarea cache-ului pentru dependințe
# Salvam pe Drive intregul director al pachetelor cumm/spconv (inclusiv binarul .so
# compilat JIT, aflat la radacina pachetului - vezi nota din celula de mai sus),
# ca sa nu mai recompilam la sesiunile viitoare (doar daca nu au fost deja restaurate)
if not JIT_CACHE_RESTORED:
    print("Se salveaza cache-ul JIT compilat pe Drive (doar prima data, poate dura cateva minute)...")
    !tar -cf {cumm_cache} -C {DIST_PACKAGES} cumm
    !tar -cf {spconv_cache} -C {DIST_PACKAGES} spconv
    print(f"Cache salvat in {CACHE_DIR}. Sesiunile viitoare cu acelasi CUDA/GPU vor sari compilarea JIT.")
else:
    print("Cache-ul era deja restaurat de pe Drive, nimic de salvat.")


Cache-ul era deja restaurat de pe Drive, nimic de salvat.


## Configurarea & Fișierele de Date
Se definesc path-urile către modelul pre-antrenat `.pt`, configurarea `.yaml` și scena de intrare `.bin` a ansamblului de puncte folosită în extragerea demo-ului.

In [5]:
INPUT_SCENE = "data/000000.bin"
PRETRAINED_MODEL = "data/student_ss_best.pt"
CONFIG_PATH = "data/distill_semantickitti.yaml"

print(f"Se foloseste scena de intrare: {INPUT_SCENE}")
print(f"Se foloseste modelul pre-antrenat: {PRETRAINED_MODEL}")
print(f"Se foloseste configuratia: {CONFIG_PATH}")

Se foloseste scena de intrare: data/000000.bin
Se foloseste modelul pre-antrenat: data/student_ss_best.pt
Se foloseste configuratia: data/distill_semantickitti.yaml


## Vizualizarea Ansamblului de Puncte Neprocesat



Mai întâi se definește o funcție ajutătoare pentru a vizualiza acele puncte.

In [6]:
# SemanticKITTI Color Map (BGR -> RGB) oficial
COLOR_MAP = {
    0: [0, 0, 0], 1: [100, 150, 245], 2: [100, 230, 245], 3: [30, 60, 150], 4: [80, 30, 180],
    5: [100, 80, 250], 6: [255, 30, 30], 7: [255, 40, 200], 8: [150, 30, 90], 9: [255, 0, 255],
    10: [255, 150, 255], 11: [75, 0, 75], 12: [175, 0, 75], 13: [255, 200, 0], 14: [255, 120, 50],
    15: [0, 175, 0], 16: [135, 60, 0], 17: [150, 240, 80], 18: [255, 240, 150], 19: [255, 0, 0],
}

# Functia care va fi folosita global pentru a genera graficul de vizualizare
def plot_point_cloud(points, colors, title, hover_texts=None, is_categorical=True):
    subsample = 10
    pts = points[::subsample]
    clrs = colors[::subsample] if not isinstance(colors, list) else [colors[i] for i in range(0, len(colors), subsample)]

    marker_dict = dict(size=2, color=clrs, opacity=0.8)
    if not is_categorical:
        marker_dict["colorscale"] = "Turbo"

    fig = go.Figure(data=[go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=marker_dict,
        text=hover_texts[::subsample] if hover_texts else None,
        hoverinfo="text" if hover_texts else "none"
    )])
    # Facem marginile mai inguste pentru a maximiza area de vizualizare 3D
    fig.update_layout(
        title=title,
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=30)
    )
    fig.show()

Înainte de a aplica inferența, se pot vizualiza punctele LiDAR neprocesate, încărcate direct din fișierul `.bin`, colorate în funcție de înălțime (axa Z).

In [7]:
raw_pc = np.fromfile(INPUT_SCENE, dtype=np.float32).reshape((-1, 4))
xyz = raw_pc[:, :3]

# Randarea punctelor neprocesate, colorate dupa axa lor Z.
hover_z = [f"Z-inaltime: {z:.2f}m" for z in xyz[:, 2]]
plot_point_cloud(xyz, xyz[:, 2], "Ansamblul de puncte LiDAR Neprocesat (colorat după înălțime)", hover_texts=hover_z, is_categorical=False)

## Încărcarea Datelor & Voxelizarea
Se folosește voxelizarea oficială `cylinder_dataset` din `Cylinder3d` pe un wrapper personalizat pentru un dataset cu o singură scenă.

In [8]:
# Deoarece noi vrem sa procesam o singura scena, construim o clasa de tip torch dataset, pentru
# o singura data neprocesata
class SinglePointCloudDataset(torch.utils.data.Dataset):
    '''
    Point Cloud class util to process single LiDAR scene.
    '''
    def __init__(self, bin_path):
        self.bin_path = bin_path
    def __len__(self):
        return 1
    def __getitem__(self, index):
        raw_data = np.fromfile(self.bin_path, dtype=np.float32).reshape((-1, 4))
        annotated_data = np.zeros((raw_data.shape[0], 1), dtype=np.int32)
        return (raw_data[:, :3], annotated_data.astype(np.uint8), raw_data[:, 3])

pt_dataset = SinglePointCloudDataset(INPUT_SCENE)

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

student_config = config["student_params"]
dataset_config = config["dataset_params"]

voxel_dataset = cylinder_dataset(
    pt_dataset,
    grid_size=student_config["output_shape"],
    fixed_volume_space=dataset_config["fixed_volume_space"],
    max_volume_space=dataset_config["max_volume_space"],
    min_volume_space=dataset_config["min_volume_space"],
    ignore_label=dataset_config["ignore_label"],
    return_test=False,
    num_scales=student_config["num_scales"]
)

batch = collate_fn_BEV([voxel_dataset[0]])
_, _, val_grid, _, val_pt_fea, val_grid_ms = batch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
pt_fea_ten = [torch.from_numpy(i).float().to(device) for i in val_pt_fea]
grid_ten = [torch.from_numpy(i).to(device) for i in val_grid]
grid_ms_ten = [torch.from_numpy(i).to(device) for i in val_grid_ms]

print(f"Norul de puncte a fost pregatit. Puncte grid: {len(grid_ten[0])}")

Norul de puncte a fost pregatit. Puncte grid: 123389


## Instanțierea Modelului & Pasul de Propagare
Se instanțiază modelul NUC-Net, se încărcă ponderile pre-antrenate și se trec datele prin model pentru a obține predicțiile de segmentare semantică.

In [9]:
model = build_student(student_config, device=str(device))

if os.path.exists(PRETRAINED_MODEL):
    state_dict = torch.load(PRETRAINED_MODEL, map_location=device)
    if "student_state_dict" in state_dict:
        state_dict = state_dict["student_state_dict"]
    model.load_state_dict(state_dict, strict=False)
    print("Modelul pre-antrenat a fost incarcat cu succes!")
else:
    print("Eroare: Modelul pre-antrenat nu a fost gasit!")

model.eval()
print("Rulare inferenta...")
start_time_nucnet = time.time()
with torch.no_grad():
    outputs = model(pt_fea_ten, grid_ten, 1, train_vox_ten_ms=grid_ms_ten)
    logits = outputs["dense_logits"] if isinstance(outputs, dict) else outputs
    predict_labels = torch.argmax(logits, dim=1).cpu().numpy()
end_time_nucnet = time.time()

per_point_pred = predict_labels[0, val_grid[0][:, 0], val_grid[0][:, 1], val_grid[0][:, 2]]
print(f"Inferenta semantica s-a incheiat! Dimensiunea etichetelor prezise: {per_point_pred.shape}")
print(f"Inferenta NUC-Net a durat: {end_time_nucnet - start_time_nucnet:.4f} secunde pe {device}")

[120 360  32]
[Student] Total params: 14.08M (trainable: 14.08M)
Modelul pre-antrenat a fost incarcat cu succes!
Rulare inferenta...


/content/libs/nucnet/models/feature_extractor.py:54: FutureWarning:

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.

W0703 17:13:40.334000 5407 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Inferenta semantica s-a incheiat! Dimensiunea etichetelor prezise: (123389,)
Inferenta NUC-Net a durat: 1.2785 secunde pe cuda:0


/usr/local/lib/python3.12/dist-packages/spconv/pytorch/core.py:127: UserWarning:

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)



## Vizualizarea Segmentării Semantice
Aici sunt prezentate clasele semantice deduse de NUC-Net Distilat. Dacă se trece cu mouse-ul peste puncte, va apărea clasa lor semantică într-un format lizibil.

In [10]:
sem_colors = [f"rgb({COLOR_MAP[l][0]},{COLOR_MAP[l][1]},{COLOR_MAP[l][2]})" for l in per_point_pred]
sem_hover = [f"Clasa: {CLASSES.get(l, 'Unknown').title()}" for l in per_point_pred]
plot_point_cloud(xyz, sem_colors, "Rezultatele Segmentarii Semantice", hover_texts=sem_hover)

## Clusterizarea Instanțelor cu Alpine
Se invocă modulul de clusterizare **Alpine**.

In [11]:
# Instantierea modului oficial Alpine
alpine_model = Alpine(THING_CLASSES, BBOX_WEB, k=32, split=False, margin=1.3)

print("Rularea Alpine...")
start_time_alpine = time.time()
inst_pred = alpine_model.fit_predict(xyz[:, :2], per_point_pred)
end_time_alpine = time.time()
print("Clusterizarea Alpine a fost finalizata.")
print(f"Clusterizarea Alpine a durat: {end_time_alpine - start_time_alpine:.4f} secunde pe {device}")

Rularea Alpine...
Clusterizarea Alpine a fost finalizata.
Clusterizarea Alpine a durat: 0.0954 secunde pe cuda:0


Culorile semantice originale pentru toate clasele 'stuff' sunt menținute, și doar culorile pentru instanțele 'thing' sunt schimbate astfel încât fiecare instanță (de exemplu, mașini individuale) primește o nuanță aleatorie plus un identificator unic pentru a le separa vizual una de alta.

In [12]:
# Gruparea instantelor dupa clasa lor semantica pentru a genera variatii a culorii de baza
from collections import defaultdict
sem_to_insts = defaultdict(set)
for sem, inst in zip(per_point_pred, inst_pred):
    if inst > 0:
        sem_to_insts[sem].add(inst)

inst_color_dict = {}
np.random.seed(42)
for sem, insts in sem_to_insts.items():
    base_color = COLOR_MAP[sem]
    for inst in insts:
        if len(insts) == 1:
            inst_color_dict[inst] = f"rgb({base_color[0]},{base_color[1]},{base_color[2]})"
        else:
            # Generarea variatiilor distincte pentru culoarea semantica de baza
            noise = np.random.randint(-50, 50, size=3)
            factor = np.random.uniform(0.6, 1.4)
            new_color = np.clip(np.array(base_color) * factor + noise, 0, 255).astype(int)
            inst_color_dict[inst] = f"rgb({new_color[0]},{new_color[1]},{new_color[2]})"

panoptic_colors = []
panoptic_hover = []

for sem, inst in zip(per_point_pred, inst_pred):
    class_name = CLASSES.get(sem, 'Unknown').title()
    if inst > 0:
        panoptic_colors.append(inst_color_dict[inst])
        panoptic_hover.append(f"Clasa: {class_name} | ID Instanta: {inst}")
    else:
        panoptic_colors.append(f"rgb({COLOR_MAP[sem][0]},{COLOR_MAP[sem][1]},{COLOR_MAP[sem][2]})")
        panoptic_hover.append(f"Clasa: {class_name}")

plot_point_cloud(xyz, panoptic_colors, "Rezultatul Final de Segmentare Panoptica Alpine", hover_texts=panoptic_hover)